In [0]:
%sql
CREATE CATALOG IF NOT EXISTS bootcamp;
--Bloque 1 — Schema:
CREATE SCHEMA IF NOT EXISTS bootcamp.practice;
--Bloque 2 — Tabla clientes:
CREATE OR REPLACE TABLE bootcamp.practice.clientes (
cliente_id INT,
nombre STRING,
email STRING,
ciudad STRING
);
INSERT INTO bootcamp.practice.clientes VALUES
(1, 'Ana', 'ana@email.com', 'Buenos Aires'),
(2, 'Carlos', 'carlos@email.com', 'Córdoba'),
(3, 'María', 'maria@email.com', 'Buenos Aires'),
(4, 'Diego', 'diego@email.com', 'Rosario'),
(5, 'Laura', 'laura@email.com', 'Mendoza'),
(6, 'Pedro', 'pedro@email.com', 'Buenos Aires'); -- Sin órdenes

CREATE OR REPLACE TABLE bootcamp.practice.productos (
producto_id INT,
nombre STRING,
categoria STRING,
precio DOUBLE
);
INSERT INTO bootcamp.practice.productos VALUES
(101, 'Laptop Pro', 'Electrónica', 1200.00),
(102, 'Mouse Inalámbrico', 'Electrónica', 35.00),
(103, 'Teclado Mecánico', 'Electrónica', 89.00),
(104, 'Monitor 27"', 'Electrónica', 450.00),
(105, 'Auriculares BT', 'Audio', 75.00),
(106, 'Webcam HD', 'Accesorios', 55.00),
(107, 'Hub USB-C', 'Accesorios', 42.00),
(108, 'Silla Gamer', 'Muebles', 380.00); -- Nunca vendida

CREATE OR REPLACE TABLE bootcamp.practice.ordenes (
orden_id INT,
cliente_id INT,
producto_id INT,
cantidad INT,
fecha DATE
);
INSERT INTO bootcamp.practice.ordenes VALUES
(1001, 1, 101, 1, '2024-01-15'),
(1002, 1, 102, 2, '2024-01-15'),
(1003, 2, 103, 1, '2024-02-01'),
(1004, 3, 101, 1, '2024-02-10'),
(1005, 3, 105, 3, '2024-02-10'),
(1006, 4, 104, 1, '2024-03-05'),
(1007, 5, 106, 2, '2024-03-12'),
(1008, 5, 107, 1, '2024-03-12'),
(1009, 2, 102, 1, '2024-04-01'),
(1010, 99, 101, 1, '2024-04-15'), -- cliente_id=99 NO existe
(1011, 1, 999, 2, '2024-05-01'); -- producto_id=999 NO existe


CREATE OR REPLACE TABLE bootcamp.practice.empleados (
empleado_id INT,
nombre STRING,
cargo STRING,
jefe_id INT
);
INSERT INTO bootcamp.practice.empleados VALUES
(1, 'Roberto', 'CEO', NULL),
(2, 'Silvia', 'Directora de Datos', 1),
(3, 'Martín', 'Director Comercial', 1),
(4, 'Lucía', 'Data Engineer Sr', 2),
(5, 'Pablo', 'Data Engineer Jr', 4),
(6, 'Camila', 'Data Analyst', 2),
(7, 'Tomás', 'Vendedor Sr', 3),
(8, 'Valentina', 'Vendedora Jr', 3);



In [0]:
%sql
USE bootcamp.practice;
select * from clientes c
join ordenes o
on c.cliente_id=o.cliente_id
order by c.cliente_id, o.orden_id;


In [0]:
%sql

select o.orden_id,
       o.fecha,
       p.nombre,
       p.categoria,
       o.cantidad,
       p.precio,
       o.cantidad*p.precio as total
from ordenes o
join productos p
on o.producto_id=p.producto_id
order by o.orden_id;

In [0]:
%sql



select c.nombre,
       c.ciudad,
       o.orden_id,
       o.fecha,
       p.nombre,
       p.categoria,
       o.cantidad,
       p.precio,
       o.cantidad*p.precio as total
from ordenes o
join productos p
on o.producto_id=p.producto_id
join clientes c
on o.cliente_id=c.cliente_id
order by o.orden_id;
    



In [0]:
%sql

select c.cliente_id,c.nombre,c.ciudad,c.email,o.* from clientes c
left join ordenes o
on c.cliente_id=o.cliente_id
order by c.cliente_id

In [0]:
%sql

select * from clientes c
left join ordenes o
on c.cliente_id=o.cliente_id
where orden_id is null
order by c.cliente_id

In [0]:
%sql

select p.*, o.* from productos p
left join ordenes o
on p.producto_id=o.producto_id


In [0]:
%sql

select p.*, o.* from productos p
left join ordenes o
on p.producto_id=o.producto_id
where orden_id is null

In [0]:
%sql

select o.*, c.* from  clientes c
right join ordenes o
on c.cliente_id=o.cliente_id

In [0]:
%sql

select o.orden_id,o.cliente_id as id_inexistente from clientes c
right join ordenes o
on c.cliente_id=o.cliente_id
where c.cliente_id is null

In [0]:
%sql

select c.*, o.* from clientes c
left join ordenes o
on c.cliente_id=o.cliente_id

In [0]:
%sql

select c.*,o.* from clientes c
full outer join ordenes o
on c.cliente_id = o.cliente_id

In [0]:
%sql
SELECT 
    COALESCE(c.cliente_id, o.cliente_id) as cliente_id,
    c.nombre as cliente,
    o.orden_id,
    CASE 
        WHEN c.cliente_id IS NOT NULL AND o.orden_id IS NOT NULL THEN '✅ Match correcto'
        WHEN c.cliente_id IS NOT NULL AND o.orden_id IS NULL THEN '⚠️ Cliente sin órdenes'
        WHEN c.cliente_id IS NULL AND o.orden_id IS NOT NULL THEN '❌ Orden huérfana'
    END as estado
FROM bootcamp.practice.clientes c
FULL OUTER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY 
    CASE 
        WHEN c.cliente_id IS NULL THEN 3  -- Huérfanas al final
        WHEN o.orden_id IS NULL THEN 2    -- Sin órdenes después
        ELSE 1                             -- Matches primero
    END,
    c.cliente_id;

In [0]:
%sql

select t1.empleado_id,t1.cargo,t1.nombre as empleado,t1.jefe_id,t2.nombre as jefe from empleados t1
left join empleados t2
on t1.jefe_id=t2.empleado_id

In [0]:
%sql

select t1.empleado_id,t1.cargo,t1.nombre as empleado,t1.jefe_id    from  empleados t1
left join empleados t2
on t1.jefe_id=t2.empleado_id
where t2.empleado_id is null

In [0]:
%sql

select jefe.cargo,jefe.nombre as Cargo,t2.nombre as empleado, t2.cargo from empleados jefe
join empleados t2
on jefe.empleado_id=t2.jefe_id


-- SELECT 
--     j.nombre as jefe,
--     j.cargo as cargo_jefe,
--     e.nombre as subordinado,
--     e.cargo as cargo_subordinado
-- FROM bootcamp.practice.empleados e
-- INNER JOIN bootcamp.practice.empleados j ON e.jefe_id = j.empleado_id
-- ORDER BY j.nombre, e.nombre;

In [0]:

%sql

select c.nombre,c.ciudad,o.orden_id,p.nombre,p.categoria,o.cantidad,p.precio, (o.cantidad*p.precio) as Total from clientes c
inner join ordenes o
on c.cliente_id=o.cliente_id
inner join productos p
on o.producto_id=p.producto_id
order by c.cliente_id,o.fecha

In [0]:
%sql

select c.nombre,c.ciudad,count(o.orden_id) as Ordenes, COALESCE(SUM(o.cantidad * p.precio), 0) as monto_total_gastado from clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
LEFT JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
ORDER BY monto_total_gastado DESC;

In [0]:
%sql

select p.nombre,p.categoria, coalesce(sum(o.cantidad),0) as Total_ventas,coalesce(sum(o.cantidad*p.precio),0) as Total_ingresos from productos p
left join ordenes o
on p.producto_id=o.producto_id
group by p.producto_id,p.nombre,p.categoria

In [0]:
%sql

select p.nombre,p.categoria, coalesce(sum(o.cantidad),0) as Total_ventas,coalesce(sum(o.cantidad*p.precio),0) as Total_ingresos from productos p
left join ordenes o
on p.producto_id=o.producto_id
group by p.nombre,p.categoria